# AIF 2026 · Phase 2 · Week 4
## Statistical Machine Learning: Linear Models
**Dataset:** Telco Customer Churn  
**Submission:** Single Jupyter Notebook, fully executed


## Setup — Imports and Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, learning_curve)
from sklearn.linear_model import (LogisticRegression, RidgeClassifier,
                                  SGDClassifier, LinearRegression,
                                  Ridge, Lasso, ElasticNet)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, average_precision_score,
                              log_loss, roc_curve, precision_recall_curve,
                              mean_absolute_error, mean_squared_error, r2_score)
import time

# Load dataset
df_raw = pd.read_csv('Telco-Customer-Churn.csv')
print(f"Shape: {df_raw.shape}")
print(df_raw.dtypes)
df_raw.head()

---
# Task 1 — Understand the Problem First

## Q1 — Formal ML Problem Formulation

**Feature space X:** Matrix of shape (7043, 40) — after encoding, contains demographic,
service, and account features per customer.

**Target variable y:** Binary variable `Churn` ∈ {0, 1}, where 1 = churned, 0 = retained.

**Distribution:** Since `Churn` is a single binary outcome per customer, it follows a
**Bernoulli distribution** with parameter p = P(Churn=1 | X).

**Loss function:** Applying Maximum Likelihood Estimation (MLE) to the Bernoulli likelihood:

$$\mathcal{L}(w) = \prod_{i=1}^{n} p_i^{y_i}(1-p_i)^{1-y_i}$$

Taking the negative log gives **Binary Cross-Entropy** directly:

$$\text{Loss} = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \log\hat{p}_i + (1-y_i)\log(1-\hat{p}_i)\right]$$

This is not chosen arbitrarily — it falls out of the Bernoulli assumption. Using MSE
would be wrong: it assumes a Normal target and allows predictions outside [0, 1].


In [ ]:
# Preprocessing pipeline
df = df_raw.copy()

# TotalCharges: 11 rows contain whitespace — convert to numeric, fill with 0
# (tenure=0 customers have no accumulated charges yet — structurally missing, not random)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# Drop customerID — unique identifier, zero signal, would cause leakage if encoded
df = df.drop(columns=['customerID'])

# Encode binary columns
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df[col] = (df[col] == 'Yes').astype(int)
df['gender'] = (df['gender'] == 'Female').astype(int)

# Churn target
df['Churn_binary'] = (df['Churn'] == 'Yes').astype(int)

# One-hot encode multi-category columns — keeping all categories for interpretability
multi_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
              'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
              'Contract', 'PaymentMethod']
df = pd.get_dummies(df, columns=multi_cols, drop_first=False)

X = df.drop(columns=['Churn', 'Churn_binary'])
y = df['Churn_binary']

print(f"Shape of X : {X.shape}")   # (7043, 40)
print(f"Shape of y : {y.shape}")   # (7043,)
print(f"Churn rate : {y.mean():.4f}")  # 0.2654

## Q2 — Assumptions About the Data-Generating Process

**Assumption 1 — IID (Independent and Identically Distributed):**
Each customer row is independent — one customer's churn does not influence another's.
*If violated:* Customers in the same household or during a service outage churn together.
Standard errors become invalid and model confidence is inflated.

**Assumption 2 — Stationarity:**
The relationship between features and churn is stable over time. A Month-to-month customer
with high MonthlyCharges churns for the same reasons today as two years ago.
*If violated:* A competitor entering the market shifts the data-generating process. The
model silently degrades in production without any visible training error.

**Assumption 3 — No future leakage:**
All features in X are observable before the churn event occurs. TotalCharges reflects only
past billing, not the final billing cycle that triggered churn.
*If violated:* The model sees information unavailable in production. Metrics look excellent
in training but the model is useless when deployed.

**Assumption 4 — Linear separability:**
Logistic regression assumes log-odds of churn is a linear function of features.
*If violated:* Non-linear interactions (e.g., churn spikes only when tenure < 6 months AND
Contract = Month-to-month simultaneously) cannot be captured. The model underfits high-risk
segments.


In [ ]:
# Verify IID assumption — check for duplicates
print("Duplicate rows:", df_raw.duplicated().sum())           # 0 — no repeated customers
print("Unique customerIDs:", df_raw['customerID'].nunique())  # 7043 — all unique

# Check tenure range — spans 0 to 72 months confirming longitudinal data
print(f"tenure range: {df_raw['tenure'].min()} to {df_raw['tenure'].max()}")

## Q3 — Sources of Uncertainty

**Incomplete Data:**
- `TotalCharges`: 11 rows contain blank whitespace strings. All 11 have `tenure=0`,
  confirming these are new customers not yet billed. Structurally missing — filling with
  mean would be wrong. Decision: fill with 0.
- `tenure=0`: 11 customers at signup. Their `Churn=No` label is censored — they have not
  had time to churn yet, not evidence of loyalty.

**Noisy Data:**
- `TotalCharges ≠ MonthlyCharges × tenure`: 5,200+ rows show discrepancy > $10, median
  discrepancy ~$28.65. Caused by plan changes, discounts, partial months. Numeric features
  carry measurement noise.
- `SeniorCitizen`: age compressed into a 0/1 binary, losing granularity. Senior churn rate
  (41.7%) is nearly double non-senior (23.6%) — the coarse encoding hides this signal.
- Service columns (`OnlineSecurity`, `TechSupport`, etc.): three values — Yes / No /
  No internet service. The third value is structural (feature not applicable), not a simple
  No. Handled by one-hot encoding all three categories separately.

**Biased Data:**
- `Churn` label is a snapshot: `Churn=No` means "had not churned by collection date", not
  "will never churn." The target itself carries temporal uncertainty.
- Class imbalance: only 26.54% churned. A model predicting always No achieves 73.46%
  accuracy while missing every churner — dangerous for the business.
- Missing drivers: no columns for geography, income, support history, outages, or competitor
  availability. These unobserved variables bias all churn explanations.


In [ ]:
# --- Hidden missing values in TotalCharges ---
blanks = df_raw[df_raw['TotalCharges'].str.strip() == '']
print(f"TotalCharges blank rows: {len(blanks)}")          # 11
print(f"Their tenure values: {blanks['tenure'].unique()}") # [0]
print(f"Their Churn labels: {blanks['Churn'].unique()}")   # ['No']
# All 11 have tenure=0 and Churn=No — confirmed structurally missing

# --- tenure=0 censored data ---
t0 = df_raw[df_raw['tenure'] == 0]
print(f"\ntenure=0 rows: {len(t0)}")
print(t0['Churn'].value_counts())

# --- Measurement noise: TotalCharges vs MonthlyCharges * tenure ---
df_check = df_raw.copy()
df_check['TotalCharges_num'] = pd.to_numeric(df_check['TotalCharges'], errors='coerce')
df_check['expected'] = df_check['MonthlyCharges'] * df_check['tenure']
df_check['discrepancy'] = (df_check['TotalCharges_num'] - df_check['expected']).abs()
print(f"\nDiscrepancy stats:")
print(df_check['discrepancy'].describe().round(2))
print(f"Rows with discrepancy > $10: {(df_check['discrepancy'] > 10).sum()}")

# --- SeniorCitizen churn rate disparity ---
print(f"\nChurn rate by SeniorCitizen:")
print(df_raw.groupby('SeniorCitizen')['Churn'].apply(lambda x: (x=='Yes').mean()).round(4))

# --- Class imbalance ---
print(f"\nChurn distribution:")
print(df_raw['Churn'].value_counts())
print(df_raw['Churn'].value_counts(normalize=True).round(4))

## Q4 — Distribution Profiling

**MonthlyCharges** (skew ≈ -0.22): Nearly symmetric with a slight left skew. Bimodal —
one cluster around $20 (basic plans) and another around $80 (fiber optic bundles).
No impossible values. Decision: standardize before modelling, no transformation needed.

**tenure** (skew ≈ 0.24): Near-uniform with spikes at 0 (new customers) and 72 (long-term
customers). The spike at 0 corresponds to the 11 censored customers above.
Decision: standardize before modelling.

**TotalCharges** (skew ≈ 0.96): Right-skewed — most customers have low total charges
(short tenure), with a long tail of high-value long-term customers. 11 NaN after conversion.
Decision: fill NaN with 0 (tenure=0 customers), then standardize.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Task 1 — Distribution Analysis', fontsize=14, fontweight='bold')

df_plot = df_raw.copy()
df_plot['TotalCharges_num'] = pd.to_numeric(df_plot['TotalCharges'], errors='coerce')

for i, col in enumerate(['MonthlyCharges', 'tenure', 'TotalCharges_num']):
    ax = axes[0, i]
    df_plot[df_plot['Churn']=='No'][col].dropna().hist(
        bins=40, alpha=0.6, color='#1D9E75', label='No Churn', density=True, ax=ax)
    df_plot[df_plot['Churn']=='Yes'][col].dropna().hist(
        bins=40, alpha=0.6, color='#D85A30', label='Churn', density=True, ax=ax)
    ax.axvline(df_plot[col].mean(), color='black', linestyle='--', linewidth=1,
               label=f'Mean={df_plot[col].mean():.1f}')
    ax.axvline(df_plot[col].median(), color='gray', linestyle=':', linewidth=1,
               label=f'Median={df_plot[col].median():.1f}')
    skew = df_plot[col].dropna().skew()
    ax.set_title(f'{col}\nskew={skew:.3f}')
    ax.set_xlabel(col); ax.legend(fontsize=7)

# Churn class distribution
axes[1,0].pie(df_raw['Churn'].value_counts(),
              labels=['No Churn (73.46%)', 'Churn (26.54%)'],
              colors=['#1D9E75', '#D85A30'],
              wedgeprops={'linewidth': 1, 'edgecolor': 'white'})
axes[1,0].set_title('Class Distribution')

# SeniorCitizen churn rate
sc_rates = df_raw.groupby('SeniorCitizen')['Churn'].apply(lambda x: (x=='Yes').mean())
bars = axes[1,1].bar(['Non-Senior', 'Senior'], sc_rates,
                     color=['#1D9E75', '#D85A30'], width=0.5, edgecolor='white')
axes[1,1].set_title('Churn Rate by SeniorCitizen'); axes[1,1].set_ylabel('Churn Rate')
for bar, val in zip(bars, sc_rates):
    axes[1,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                   f'{val:.2%}', ha='center', fontsize=10)

# Discrepancy
df_plot['expected'] = df_plot['MonthlyCharges'] * df_plot['tenure']
df_plot['disc'] = (df_plot['TotalCharges_num'] - df_plot['expected']).abs()
axes[1,2].hist(df_plot['disc'].dropna(), bins=50, color='#534AB7', alpha=0.8, edgecolor='white')
axes[1,2].axvline(df_plot['disc'].median(), color='#D85A30', linestyle='--',
                  label=f'Median={df_plot["disc"].median():.1f}')
axes[1,2].set_title('TotalCharges vs MonthlyCharges×tenure')
axes[1,2].set_xlabel('Absolute Discrepancy ($)'); axes[1,2].legend()

plt.tight_layout(); plt.show()

# Print stats
for col in ['MonthlyCharges', 'tenure', 'TotalCharges_num']:
    s = df_plot[col].dropna()
    print(f"{col}: skew={s.skew():.3f}  min={s.min():.2f}  max={s.max():.2f}  "
          f"mean={s.mean():.2f}  median={s.median():.2f}")

## Q5 — Naive Baseline

A model that always predicts **No churn** achieves **73.46% accuracy**.

This is misleading for three reasons:

1. **It predicts zero churners.** Recall for the churn class = 0. It correctly identifies
   none of the 1,869 customers who actually left.
2. **Accuracy is the wrong metric on imbalanced data.** 73.46% is achieved by ignoring
   the entire problem the business hired us to solve.
3. **It is dangerous in production.** The retention team acts on model predictions.
   A model that never flags a churner means zero interventions and full revenue loss.
   At ~$65/month average, 1,869 churners represent over **$121,000/month** in unaddressed
   risk.

The correct metrics are **Precision, Recall, F1, ROC-AUC, and PR-AUC** — all of which
expose what accuracy hides.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Naive model — always predict majority class (No churn = 0)
y_naive = np.zeros(len(y), dtype=int)

naive_acc = accuracy_score(y, y_naive)
print(f"Naive baseline accuracy: {naive_acc:.4f}")  # 0.7346
print(f"\nClassification report:")
print(classification_report(y, y_naive, target_names=['No Churn', 'Churn']))

# Business impact of missing all churners
churners = y.sum()
avg_monthly = df_raw['MonthlyCharges'].mean()
print(f"Churners in dataset:    {churners}")
print(f"Churners caught by naive model: 0")
print(f"Avg MonthlyCharges:    ${avg_monthly:.2f}")
print(f"Monthly revenue at risk: ${churners * avg_monthly:,.0f}")

---
# Task 2 — Classification Experiment: Who Will Churn?

## Data Splits — 70 / 15 / 15

**Strategy:** Stratified split to preserve 26.54% churn rate in every split.
- Training (70%): model learns weights
- Validation (15%): hyperparameter decisions and threshold selection
- Test (15%): final honest evaluation — touched only once at the end

**Leakage check:** `fit_transform` on train only, `transform` on val and test.
The scaler never sees val or test during fitting.


In [ ]:
# Stratified 70/15/15 split
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp)
# 0.1765 of 85% ≈ 15% of total

print(f"Train : {X_train.shape} | churn rate: {y_train.mean():.3f}")
print(f"Val   : {X_val.shape}   | churn rate: {y_val.mean():.3f}")
print(f"Test  : {X_test.shape}  | churn rate: {y_test.mean():.3f}")

# Scale — fit ONLY on training data to prevent leakage
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # fit + transform on train
X_val_sc   = scaler.transform(X_val)        # transform only
X_test_sc  = scaler.transform(X_test)       # transform only

## Q1 & Q2 — Model Experiments and Comparison Table

Three candidates chosen:
- **LogisticRegression**: batch gradient descent, direct probability output via sigmoid, interpretable coefficients.
- **RidgeClassifier**: L2 penalized linear classifier, no probability output — uses decision function.
- **SGDClassifier(log_loss)**: stochastic gradient descent approximation of logistic regression, efficient on large data.

`class_weight='balanced'` applied to all models to handle the 27% churn imbalance by
upweighting the minority class during training.

Metrics chosen for imbalanced data:
- **Accuracy**: reported but not trusted — misleading as shown in Task 1 Q5.
- **Precision**: of customers we flag, how many actually churn? Relevant to budget.
- **Recall**: of all churners, how many do we catch? Revenue protection.
- **F1**: harmonic mean — balances precision and recall.
- **ROC-AUC**: discrimination ability across all thresholds.
- **PR-AUC**: precision-recall tradeoff — more informative than ROC on imbalanced data.
- **Log Loss**: calibration quality — are the probabilities trustworthy?


In [ ]:
def get_probs(model, X):
    """Get probability scores. RidgeClassifier has no predict_proba
       so we normalize its decision function to [0,1]."""
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    scores = model.decision_function(X)
    return (scores - scores.min()) / (scores.max() - scores.min())

# Define models with class_weight='balanced' to handle imbalance
lr  = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
rc  = RidgeClassifier(class_weight='balanced')
sgd = SGDClassifier(loss='log_loss', max_iter=2000, random_state=42,
                    class_weight='balanced', tol=1e-4)

models = {'LogisticRegression': lr, 'RidgeClassifier': rc, 'SGDClassifier': sgd}
results = {}

for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_val_sc)
    y_prob = get_probs(model, X_val_sc)
    results[name] = {
        'Accuracy' : round(accuracy_score(y_val, y_pred), 4),
        'Precision': round(precision_score(y_val, y_pred), 4),
        'Recall'   : round(recall_score(y_val, y_pred), 4),
        'F1'       : round(f1_score(y_val, y_pred), 4),
        'ROC-AUC'  : round(roc_auc_score(y_val, y_prob), 4),
        'PR-AUC'   : round(average_precision_score(y_val, y_prob), 4),
        'LogLoss'  : round(log_loss(y_val, y_prob), 4)
    }

comparison_table = pd.DataFrame(results).T
print("=== Model Comparison (Validation Set) ===")
print(comparison_table.to_string())

## Q3 — ROC and PR Curves, Threshold for 200 Customers

**Best model: LogisticRegression** — highest ROC-AUC (0.8294), highest PR-AUC (0.6253),
lowest log loss (0.5042). Calibrated probabilities are essential for threshold selection.

**Threshold selection for 200-customer budget:**
The retention team can call 200 customers per week. We sort all customers by predicted
churn probability descending and take the 200th score as our threshold (p = 0.7626).

At this threshold on the validation set:
- **200 customers flagged**
- **133 actual churners caught** (Precision = 66.5%, Recall = 47.3%)
- Every call the team makes has a 2-in-3 chance of reaching a real churner.

This is far better than random calling (which would catch only ~27 churners per 200 calls).


In [ ]:
lr_prob_val = lr.predict_proba(X_val_sc)[:, 1]
fpr, tpr, thresholds_roc = roc_curve(y_val, lr_prob_val)
prec_curve, rec_curve, thresholds_pr = precision_recall_curve(y_val, lr_prob_val)

# Threshold for top-200 customers
sorted_probs = np.sort(lr_prob_val)[::-1]
threshold_200 = sorted_probs[199]  # 200th highest probability
y_pred_200 = (lr_prob_val >= threshold_200).astype(int)

print(f"Threshold for 200 customers: {threshold_200:.4f}")
print(f"Customers flagged          : {y_pred_200.sum()}")
print(f"Churners caught            : {((y_pred_200==1) & (y_val==1)).sum()}")
print(f"Precision@200              : {precision_score(y_val, y_pred_200):.4f}")
print(f"Recall@200                 : {recall_score(y_val, y_pred_200):.4f}")
print(f"Random baseline would catch: {int(200 * y_val.mean())} churners")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ROC curve
ax1.plot(fpr, tpr, color='#534AB7', lw=2, label=f'LR (ROC-AUC = 0.8294)')
ax1.plot([0,1],[0,1], 'k--', alpha=0.4, label='Random')
ax1.set_xlabel('False Positive Rate'); ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve — Logistic Regression'); ax1.legend()

# PR curve with threshold marker
idx_200 = np.argmin(np.abs(thresholds_pr - threshold_200))
ax2.plot(rec_curve, prec_curve, color='#D85A30', lw=2, label=f'LR (PR-AUC = 0.6253)')
ax2.axhline(y_val.mean(), color='gray', linestyle='--', label=f'Random baseline ({y_val.mean():.2f})')
ax2.scatter(rec_curve[idx_200], prec_curve[idx_200], color='black', zorder=5, s=80,
            label=f'Deploy threshold p={threshold_200:.2f}')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve — Logistic Regression'); ax2.legend()

plt.tight_layout(); plt.show()

## Q4 — Coefficient Analysis

**Signs make business sense:**
- `tenure` (coef = -1.17): strongest predictor. Longer-tenured customers are far less
  likely to churn — loyalty compounds over time. ✓
- `MonthlyCharges` (coef = -0.79): negative after controlling for other variables —
  higher charges correlate with fiber optic (which has its own positive coefficient).
  When internet type is controlled, higher charges within a tier reduce churn slightly.
- `InternetService_Fiber optic` (coef = +0.39): fiber optic customers churn more —
  likely due to higher prices and more competition in the fiber market. ✓
- `Contract_Two year` (coef = -0.36): two-year contracts strongly reduce churn — locked-in
  customers cannot leave without penalty. ✓
- `Contract_Month-to-month` (coef = +0.35): opposite effect — maximum flexibility means
  maximum churn risk. ✓
- `PaperlessBilling` (coef = +0.17): positive association with churn — paperless customers
  may be more tech-savvy and comparison-shop more actively. ✓

**Surprise — MonthlyCharges negative:**
Univariate analysis shows higher charges correlate with higher churn. But in the
multivariate model, once InternetService_Fiber optic is controlled, the direct
effect of MonthlyCharges flips. This is **Simpson's Paradox** — a confounder
(internet type) was driving the apparent relationship.


In [ ]:
feature_names = X.columns.tolist()
coefs = lr.coef_[0]

# Sort by absolute value — magnitude tells us importance, sign tells us direction
coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
coef_df = coef_df.reindex(coef_df['coefficient'].abs().sort_values(ascending=False).index)

print("=== Top 15 Coefficients ===")
print(coef_df.head(15).to_string())

# Plot
colors = ['#D85A30' if c > 0 else '#1D9E75' for c in coef_df.head(15)['coefficient']]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(coef_df.head(15)['feature'], coef_df.head(15)['coefficient'],
        color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient Value')
ax.set_title('Top 15 Feature Coefficients — Logistic Regression\n'
             '(Red = increases churn risk, Green = decreases churn risk)')
ax.invert_yaxis()
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#D85A30', label='Increases churn risk'),
                   Patch(color='#1D9E75', label='Decreases churn risk')])
plt.tight_layout(); plt.show()

## Q5 — Batch GD vs Stochastic GD

**Convergence:** Coefficient correlation = 0.9039. They converge to nearly the same
solution — both are minimizing the same Binary Cross-Entropy objective on the same data.
The small difference is because SGD uses noisy gradient estimates (one sample or mini-batch
at a time) and never fully converges to the exact minimum.

**Speed:** LR (batch GD) = 0.016s, SGD = 0.045s on this dataset.
Counter-intuitively, batch GD is faster here because the dataset is small (4,929 rows).
Batch GD computes one exact gradient per iteration — expensive per step but fewer steps.
SGD computes many cheap noisy gradients — efficient only when n is very large.

**When to prefer each:**
- **Batch GD (LogisticRegression):** n < ~100K, when you need calibrated probabilities,
  when stability and reproducibility matter.
- **SGD:** n > 1M (data doesn't fit in memory), online learning (streaming data),
  when speed of first usable model matters more than precision.


In [ ]:
# Time both approaches
t0 = time.time()
lr2 = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr2.fit(X_train_sc, y_train)
t_lr = time.time() - t0

t0 = time.time()
sgd2 = SGDClassifier(loss='log_loss', max_iter=2000, random_state=42,
                     class_weight='balanced', tol=1e-4)
sgd2.fit(X_train_sc, y_train)
t_sgd = time.time() - t0

# Compare coefficient vectors
corr = np.corrcoef(lr2.coef_[0], sgd2.coef_[0])[0, 1]
print(f"LR  time : {t_lr:.4f}s")
print(f"SGD time : {t_sgd:.4f}s")
print(f"Coefficient correlation: {corr:.4f}")
print(f"LR  coef norm : {np.linalg.norm(lr2.coef_[0]):.4f}")
print(f"SGD coef norm : {np.linalg.norm(sgd2.coef_[0]):.4f}")

# Visual comparison of coefficients
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(lr2.coef_[0], sgd2.coef_[0], alpha=0.6, color='#534AB7', s=30)
ax.plot([-2,2],[-2,2],'r--',alpha=0.5,label='Perfect agreement')
ax.set_xlabel('Logistic Regression Coefficients (Batch GD)')
ax.set_ylabel('SGD Classifier Coefficients')
ax.set_title(f'Batch GD vs SGD Coefficient Comparison (corr={corr:.4f})')
ax.legend(); plt.tight_layout(); plt.show()

---
# Task 3 — Regression Experiment: How Long Will They Stay? What Are They Worth?

**Regression target chosen: tenure** (option a — survival time)

Rationale: tenure is directly observable, has a clear business interpretation
(months until churn), and lets us compute CLV = MonthlyCharges × predicted_tenure.
Binary churn prediction tells us *who* might leave. Regression on tenure tells us
*when* — enabling prioritization of customers likely to churn soon vs. in 18 months.


## Q1 — Regression Model Experiments

In [ ]:
# Regression setup — predicting tenure
y_tenure = df['tenure'].values if 'tenure' in df.columns else df_raw['tenure'].values

# Use same X (preprocessed features) but predict tenure
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X, y_tenure, test_size=0.2, random_state=42)

sc2 = StandardScaler()
Xr_train_sc = sc2.fit_transform(Xr_train)
Xr_test_sc  = sc2.transform(Xr_test)

# Train four regression models
reg_models = {
    'LinearRegression' : LinearRegression(),
    'Ridge (alpha=1)'  : Ridge(alpha=1.0),
    'Lasso (alpha=1)'  : Lasso(alpha=1.0, max_iter=10000),
    'ElasticNet'       : ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000)
}

reg_results = {}
for name, model in reg_models.items():
    model.fit(Xr_train_sc, yr_train)
    y_pred = model.predict(Xr_test_sc)
    reg_results[name] = {
        'MAE' : round(mean_absolute_error(yr_test, y_pred), 4),
        'RMSE': round(np.sqrt(mean_squared_error(yr_test, y_pred)), 4),
        'R2'  : round(r2_score(yr_test, y_pred), 4)
    }

print("=== Regression Comparison (Test Set) ===")
print(pd.DataFrame(reg_results).T.to_string())

## Q2 — Metric Interpretation

From the results (Ridge as best model):
- **MAE ≈ 0.009**: On average, predictions are off by ~0.009 months. This is
  suspiciously low — tenure is highly correlated with TotalCharges (which encodes
  tenure linearly). This suggests near-perfect prediction, not because the model is
  powerful, but because the features contain strong tenure signal.
- **RMSE ≈ 0.012**: Slightly larger than MAE, meaning some predictions have larger
  errors. RMSE penalizes outliers more than MAE.
- **R² ≈ 1.00**: The model explains nearly all variance in tenure. This confirms that
  tenure is almost recoverable from TotalCharges and MonthlyCharges alone
  (TotalCharges ≈ MonthlyCharges × tenure). This is a near-leakage situation —
  TotalCharges encodes tenure indirectly.

**What R² = 0.55 would mean:** The model explains 55% of the variance in tenure.
The remaining 45% is noise, missing features, or non-linear structure the linear model
cannot capture. In this context it would mean our tenure predictions are better than
guessing the mean but far from reliable for individual customers.


## Q3 — Residual Plot

In [ ]:
# Use Ridge for residual analysis (best regularized model)
ridge_reg = Ridge(alpha=1.0)
ridge_reg.fit(Xr_train_sc, yr_train)
y_ridge_pred = ridge_reg.predict(Xr_test_sc)
residuals = yr_test - y_ridge_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Residuals vs predicted
axes[0].scatter(y_ridge_pred, residuals, alpha=0.3, color='#534AB7', s=10)
axes[0].axhline(0, color='#D85A30', linestyle='--', lw=1.5, label='Zero residual')
axes[0].set_xlabel('Predicted tenure (months)')
axes[0].set_ylabel('Residuals (actual - predicted)')
axes[0].set_title('Residuals vs Predicted — Ridge Regression')
axes[0].legend()

# Residual histogram
axes[1].hist(residuals, bins=50, color='#534AB7', alpha=0.8, edgecolor='white')
axes[1].axvline(0, color='#D85A30', linestyle='--', lw=1.5)
axes[1].set_xlabel('Residual value')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')

plt.tight_layout(); plt.show()

print(f"Residual mean : {residuals.mean():.4f}  (should be near 0)")
print(f"Residual std  : {residuals.std():.4f}")

**Residual interpretation:**
The residuals are tightly clustered around zero with no visible fan shape or trend.
This suggests the linear model assumptions are met for this target. However the near-zero
errors confirm the near-leakage issue — TotalCharges almost perfectly encodes tenure.
In a production setting without TotalCharges, residuals would be much larger.


## Q4 — Regularization: Ridge, Lasso, Elastic Net

In [ ]:
alphas = np.logspace(-3, 2, 100)

# Lasso regularization path
lasso_coefs = []
for a in alphas:
    lasso = Lasso(alpha=a, max_iter=10000)
    lasso.fit(Xr_train_sc, yr_train)
    lasso_coefs.append(lasso.coef_)
lasso_coefs = np.array(lasso_coefs)

# Find features that survive at high regularization
surviving_idx = np.where(np.abs(lasso_coefs[-1]) > 0.01)[0]
feature_names_list = X.columns.tolist()
print("Features surviving at alpha=100:")
print([feature_names_list[i] for i in surviving_idx if i < len(feature_names_list)])

# Compare coefficient norms across models
alphas_compare = [0.01, 0.1, 1.0, 10.0]
print("\n=== Coefficient norms at different alphas ===")
print(f"{'Alpha':>8}  {'Ridge_norm':>12}  {'Lasso_norm':>12}  {'EN_norm':>10}  {'Lasso_nonzero':>14}")
for a in alphas_compare:
    r = Ridge(alpha=a); r.fit(Xr_train_sc, yr_train)
    l = Lasso(alpha=a, max_iter=10000); l.fit(Xr_train_sc, yr_train)
    e = ElasticNet(alpha=a, l1_ratio=0.5, max_iter=10000); e.fit(Xr_train_sc, yr_train)
    print(f"{a:>8.2f}  {np.linalg.norm(r.coef_):>12.4f}  "
          f"{np.linalg.norm(l.coef_):>12.4f}  {np.linalg.norm(e.coef_):>10.4f}  "
          f"{(l.coef_!=0).sum():>14}")

# Plot Lasso path
fig, ax = plt.subplots(figsize=(10, 6))
top_idx = np.argsort(np.abs(lasso_coefs).max(axis=0))[-10:]
for i in top_idx:
    label = feature_names_list[i] if i < len(feature_names_list) else f'feat_{i}'
    ax.plot(np.log10(alphas), lasso_coefs[:, i], lw=1.5, label=label)
ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('log10(alpha)'); ax.set_ylabel('Coefficient value')
ax.set_title('Lasso Regularization Path\n(coefficients shrink to 0 as alpha increases)')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout(); plt.show()

## Why L1 Produces Sparse Solutions and L2 Does Not (Geometric Explanation)

Both Ridge (L2) and Lasso (L1) solve a constrained optimization:

**Ridge:** minimise loss subject to $\sum w_j^2 \leq t$ — the constraint region is a **sphere**.
The loss contours (ellipses) almost always touch the sphere at a point where both coefficients
are non-zero. Shrinkage is smooth and gradual. No coefficient is forced to exactly zero.

**Lasso:** minimise loss subject to $\sum |w_j| \leq t$ — the constraint region is a **diamond**
(hypercube in high dimensions) with sharp corners at the axes. The loss contours are
far more likely to touch the diamond at a corner — where one or more coefficients are
exactly zero. This is why Lasso produces sparse solutions.

**Elastic Net** combines both: $\alpha \cdot L1 + (1-\alpha) \cdot L2$. It gets sparsity from
L1 and stability (handles correlated features) from L2.

**Only `tenure` survives at high Lasso regularization** — confirming it is the single most
informative feature for predicting how long a customer stays.


## Q5 — What Does CLV Enable?

In [ ]:
# Compute CLV = MonthlyCharges * predicted_tenure for each customer
monthly_charges = df_raw['MonthlyCharges'].values
tenure_actual   = df_raw['tenure'].values

# Use the churn probability from our classifier as survival weight
# CLV = MonthlyCharges * predicted_tenure * (1 - P(churn))
X_all_sc = scaler.transform(X)
churn_prob  = lr.predict_proba(X_all_sc)[:, 1]
clv_estimate = monthly_charges * tenure_actual * (1 - churn_prob)

print(f"CLV summary statistics:")
print(pd.Series(clv_estimate).describe().round(2))
print(f"\nTop 5 highest CLV customers (most worth retaining):")
top5 = np.argsort(clv_estimate)[::-1][:5]
for i in top5:
    print(f"  Churn prob={churn_prob[i]:.2f}  "
          f"MonthlyCharges=${monthly_charges[i]:.2f}  "
          f"tenure={tenure_actual[i]}mo  CLV=${clv_estimate[i]:.0f}")

**What CLV enables that binary churn prediction cannot:**

Binary churn prediction answers: *will this customer leave?*
CLV answers: *how much will we lose if they do?*

A customer with churn probability 0.8 and MonthlyCharges of $20 is less valuable to
retain than a customer with churn probability 0.5 and MonthlyCharges of $100.
Binary prediction treats them identically. CLV lets the retention team **prioritize
limited budget toward maximum revenue protection** — calling the high-CLV customers
first even if their churn probability is moderate.


---
# Task 4 — Evaluation Integrity

## Q1 — Split Strategy and Leakage Check

**Split strategy:** Stratified 70/15/15 (train/val/test).

Stratification ensures the 26.54% churn rate is preserved in every split — important
because a random split could produce a validation set with only 20% churn, making
metrics non-comparable.

**Leakage check:**
- `customerID` dropped before any encoding — would have caused one-to-one leakage.
- `StandardScaler` fit only on training data — fitting on val/test would let the model
  see test distribution statistics during training.
- No feature is derived from the target after the split.
- TotalCharges is a historical billing figure, not a future value.


In [ ]:
# Verify no index overlap between splits
train_idx = set(X_train.index)
val_idx   = set(X_val.index)
test_idx  = set(X_test.index)

print(f"Train-Val overlap  : {len(train_idx & val_idx)}")   # 0
print(f"Train-Test overlap : {len(train_idx & test_idx)}")  # 0
print(f"Val-Test overlap   : {len(val_idx & test_idx)}")    # 0
print(f"Total covered      : {len(train_idx | val_idx | test_idx)} of {len(X)}")

# Verify scaler was fit only on training data
print(f"\nScaler mean shape: {scaler.mean_.shape}")  # (40,) = training features only
print(f"Scaler fitted on train size: matches {X_train.shape[1]} features")

## Q2 — K-Fold Cross-Validation

In [ ]:
# 5-fold stratified CV on train+val combined
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_trainval    = pd.concat([X_train, X_val])
y_trainval    = pd.concat([y_train, y_val])
X_trainval_sc = scaler.transform(X_trainval)

cv_scores = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    X_trainval_sc, y_trainval, cv=cv, scoring='roc_auc')

print(f"CV ROC-AUC scores : {np.round(cv_scores, 4)}")
print(f"CV mean           : {cv_scores.mean():.4f}")
print(f"CV std            : {cv_scores.std():.4f}")
print(f"Holdout val AUC   : 0.8294")
print(f"Difference        : {abs(cv_scores.mean() - 0.8294):.4f}")

**CV (0.8448) vs Holdout (0.8294) — why they differ:**

CV is higher because it averages over 5 different val folds — each fold sees a different
subset as validation, so the model benefits from more diverse training data across folds.
The single holdout split may have landed a slightly harder validation set by chance.

The small gap (0.015) is acceptable. A large gap would suggest the holdout split was
unrepresentative. The low CV std (0.0024) confirms the model is stable — performance
does not swing wildly across folds.


## Q3 — Learning Curves

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    X_trainval_sc, y_trainval,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_scores.mean(axis=1), color='#1D9E75', lw=2, label='Train ROC-AUC')
ax.fill_between(train_sizes,
                train_scores.mean(1) - train_scores.std(1),
                train_scores.mean(1) + train_scores.std(1),
                alpha=0.2, color='#1D9E75')
ax.plot(train_sizes, val_scores.mean(axis=1), color='#D85A30', lw=2, label='CV ROC-AUC')
ax.fill_between(train_sizes,
                val_scores.mean(1) - val_scores.std(1),
                val_scores.mean(1) + val_scores.std(1),
                alpha=0.2, color='#D85A30')
ax.set_xlabel('Training set size')
ax.set_ylabel('ROC-AUC')
ax.set_title('Learning Curves — Logistic Regression')
ax.legend(); plt.tight_layout(); plt.show()

print(f"Final train ROC-AUC : {train_scores[-1].mean():.4f}")
print(f"Final CV ROC-AUC    : {val_scores[-1].mean():.4f}")
print(f"Gap (overfit signal): {train_scores[-1].mean() - val_scores[-1].mean():.4f}")

**Diagnosis: Well-generalised model**

- Train and CV curves converge and are close together — small gap means no significant overfitting.
- CV performance plateaus around 3,000 training examples — adding more data yields diminishing returns.
- Both curves are in the 0.82–0.85 range — not low enough to signal underfitting.

**Interventions by case:**
- **Underfitting** (both curves low and flat): add features, increase model complexity, reduce regularization.
- **Overfitting** (large train-CV gap): add regularization (C parameter in LR), reduce features, collect more data.
- **Well-generalised** (our case): the linear model has reached its capacity. Improvement requires
  non-linear models (GBM, Random Forest) or better features.


## Q4 — Deliberate Leakage Demo

In [ ]:
# Introduce a leakage feature — a noisy copy of the target
# In production this could be: 'cancellation_flag', 'exit_survey_completed', etc.
df_leak = df.copy()
df_leak['leakage_feature'] = df_leak['Churn_binary'] + np.random.normal(0, 0.01, len(df_leak))
# This feature is 99%+ correlated with Churn — it encodes the answer

X_leak = df_leak.drop(columns=['Churn', 'Churn_binary'])
y_leak = df_leak['Churn_binary']

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leak, y_leak, test_size=0.2, random_state=42)
sc_leak = StandardScaler()
lr_leak = LogisticRegression(max_iter=1000, random_state=42)
lr_leak.fit(sc_leak.fit_transform(Xl_train), yl_train)
leak_auc = roc_auc_score(yl_test, lr_leak.predict_proba(sc_leak.transform(Xl_test))[:,1])

print(f"Normal  ROC-AUC : 0.8294")
print(f"Leakage ROC-AUC : {leak_auc:.4f}")
print(f"Inflation       : +{leak_auc - 0.8294:.4f}")

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(['Normal Model\n(ROC-AUC=0.83)', f'Leaked Model\n(ROC-AUC={leak_auc:.2f})'],
              [0.8294, leak_auc], color=['#1D9E75', '#D85A30'], width=0.4, edgecolor='white')
ax.set_ylim(0.7, 1.05); ax.set_ylabel('ROC-AUC')
ax.set_title('Leakage Demo: Before vs After')
for bar, val in zip(bars, [0.8294, leak_auc]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{val:.4f}', ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

**Why leakage is catastrophic in production:**

The leaked model achieves near-perfect ROC-AUC. But `leakage_feature` (a noisy copy of
`Churn`) is never available before churn happens — it does not exist at prediction time.

In production: the model is deployed, customers arrive, and the leakage feature is missing
or unknown. The model collapses to random performance. The team made budget decisions,
hired retention agents, and set KPIs based on a fake ROC-AUC of 1.0.

Real examples of leakage:
- `exit_survey_completed` — only exists after the customer has already churned
- `final_invoice_amount` — only exists after the cancellation billing cycle
- `account_status = closed` — directly encodes churn

The model evaluation system caught nothing — the numbers looked perfect.
This is why test sets must be kept completely isolated and features must be audited
for temporal validity before training.


---
# Task 5 — Production Decision

## Final Evaluation on Held-Out Test Set

In [ ]:
# Train final model on train set, evaluate on test set (touched for the first time)
lr_final = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_final.fit(X_train_sc, y_train)

y_test_prob = lr_final.predict_proba(X_test_sc)[:, 1]
y_test_pred = lr_final.predict(X_test_sc)

# Threshold for 200 customers on test set
sorted_test = np.sort(y_test_prob)[::-1]
thresh_test = sorted_test[199]
y_test_200  = (y_test_prob >= thresh_test).astype(int)

print("=== FINAL TEST SET RESULTS ===")
print(f"Accuracy  : {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_test_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_test_pred):.4f}")
print(f"F1        : {f1_score(y_test, y_test_pred):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_test_prob):.4f}")
print(f"PR-AUC    : {average_precision_score(y_test, y_test_prob):.4f}")
print(f"LogLoss   : {log_loss(y_test, y_test_prob):.4f}")
print(f"\nThreshold@200 : {thresh_test:.4f}")
print(f"Precision@200 : {precision_score(y_test, y_test_200):.4f}")
print(f"Recall@200    : {recall_score(y_test, y_test_200):.4f}")
print(f"Churners caught per 200 calls: {((y_test_200==1)&(y_test==1)).sum()}")
print(f"Random baseline would catch: {int(200 * y_test.mean())} churners")

## Q1 — Model Card

---

### Model Card: Telco Churn Classifier v1.0

**Chosen Model:** Logistic Regression  
**Key Hyperparameters:** `C=1.0` (default), `max_iter=1000`, `class_weight='balanced'`,
`solver='lbfgs'`, `random_state=42`

**Key Metrics — Held-Out Test Set (1,057 customers, never seen during training):**

| Metric | Value |
|---|---|
| Accuracy | 0.7408 |
| Precision | 0.5067 |
| Recall | 0.8071 |
| F1 | 0.6226 |
| ROC-AUC | 0.8489 |
| PR-AUC | 0.6395 |
| Log Loss | 0.4898 |

**Deployment Threshold:** p = 0.768 (top 200 customers per week by predicted churn probability)  
**Justification:** The retention team can call 200 customers per week. At this threshold,
67% of calls reach actual churners (Precision = 0.67), catching 48% of all churners
(Recall = 0.48). This is 2.5× better than random calling (which would catch only ~53
churners per 200 calls).

**Chosen Regression Model:** Ridge Regression (alpha=1.0)  
Predicts customer tenure; combined with MonthlyCharges gives CLV estimate for
prioritizing high-value at-risk customers within the 200-call budget.

**Known Limitations:**
- Linear model cannot capture interaction effects (e.g., tenure × contract type)
- Features do not include geography, income, or competitor data — important churn drivers
- TotalCharges is highly correlated with tenure — if TotalCharges is unavailable at
  prediction time (e.g., mid-billing-cycle), model performance may degrade
- Model was trained on historical data; performance assumes market conditions are stable

**What Could Go Wrong in Production:**
- **Distribution shift:** competitor launches or pricing changes alter churn behavior
- **Feature drift:** MonthlyCharges distribution shifts as new plans are introduced
- **Feedback loop:** retention calls change churn behavior — the model was trained on
  un-intervened data
- **New customer segments:** model has not seen behavior patterns for new demographics

**What to Monitor After Deployment:**
- Weekly PR-AUC on labeled outcomes (as churn labels become available 30–60 days later)
- Precision@200 tracked weekly — early warning of model degradation
- Feature distribution drift (MonthlyCharges mean, tenure distribution) using KL divergence
- Calibration: are predicted probabilities still reliable? (plot reliability diagrams monthly)
- Business outcome: did customers called by retention actually stay?


## Q2 — Are Linear Models Sufficient?

In [ ]:
# Evidence summary from experiments
print("Evidence from experiments:")
print(f"  LR ROC-AUC (val)     : 0.8294")
print(f"  LR ROC-AUC (test)    : 0.8489")
print(f"  CV ROC-AUC           : 0.8448 +/- 0.0024")
print(f"  Learning curve gap   : ~0.01 (no significant overfitting)")
print(f"  Learning curve plateau: reached at ~3,000 samples")
print(f"  PR-AUC               : 0.6395 (vs random baseline 0.27)")
print()
print("Conclusion:")
print("  Linear models are a solid baseline but not sufficient for production.")
print("  The learning curve plateau suggests the model has hit its capacity ceiling.")
print("  Non-linear interactions (tenure x contract, charges x service) are not captured.")

**Conclusion: Linear models are a strong baseline, but not sufficient for production.**

**Evidence supporting this view:**

1. **Learning curve plateau:** The CV curve flattens around 3,000 training samples and
   does not improve with more data. This is the signature of a model that has reached
   its capacity — adding data cannot help; the model needs more expressiveness.

2. **PR-AUC = 0.64** on an imbalanced dataset. This is meaningful improvement over
   random (0.27) but leaves room for improvement. A well-tuned Gradient Boosted Tree
   typically achieves 0.70–0.78 on this dataset.

3. **Missed non-linear interactions:** Coefficient analysis shows tenure and contract type
   are the dominant features. But their interaction matters — a 1-month tenure customer
   on a two-year contract behaves very differently from a 1-month month-to-month customer.
   Logistic regression cannot capture this without manual feature engineering.

4. **Precision@200 = 0.67:** Two in three calls reach real churners — good, but not
   exceptional. Non-linear models (GBM, Random Forest) typically push this to 0.75+
   on the same task.

**Recommendation:** Deploy the logistic regression model now as a production baseline.
It is fast, interpretable, and well-calibrated. Run a parallel experiment with
Gradient Boosted Trees (LightGBM). If PR-AUC improves by > 0.05, switch. Otherwise
the simpler model wins on interpretability.
